
# **Practical 7
Aim: - Perform Clustering with PySpark.**

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans


In [2]:
def main():
    # 1. Create Spark Session
    spark = SparkSession.builder \
        .appName("PySpark Clustering Example") \
        .getOrCreate()

    # 2. Sample Dataset
    data = [
        (1, 20000, 3),
        (2, 30000, 4),
        (3, 40000, 2),
        (4, 50000, 5),
        (5, 60000, 7),
        (6, 70000, 8)
    ]

    columns = ["id", "income", "spending_score"]

    df = spark.createDataFrame(data, columns)

    print("Original Data:")
    df.show()

    # 3. Feature Vector Creation
    assembler = VectorAssembler(
        inputCols=["income", "spending_score"],
        outputCol="features"
    )

    df_vector = assembler.transform(df)

    print("Feature Vector:")
    df_vector.select("features").show(truncate=False)

    # 4. Apply K-Means Clustering
    kmeans = KMeans(k=2, seed=1)

    model = kmeans.fit(df_vector)

    # 5. Predictions
    predictions = model.transform(df_vector)

    print("Clustered Data:")
    predictions.select("id", "income", "spending_score", "prediction").show()

    # 6. Cluster Centers
    print("Cluster Centers:")
    centers = model.clusterCenters()
    for i, center in enumerate(centers):
        print(f"Cluster {i}: {center}")

    # Stop Spark Session
    spark.stop()


In [3]:

# Run the program
if __name__ == "__main__":
    main()

Original Data:
+---+------+--------------+
| id|income|spending_score|
+---+------+--------------+
|  1| 20000|             3|
|  2| 30000|             4|
|  3| 40000|             2|
|  4| 50000|             5|
|  5| 60000|             7|
|  6| 70000|             8|
+---+------+--------------+

Feature Vector:
+-------------+
|features     |
+-------------+
|[20000.0,3.0]|
|[30000.0,4.0]|
|[40000.0,2.0]|
|[50000.0,5.0]|
|[60000.0,7.0]|
|[70000.0,8.0]|
+-------------+

Clustered Data:
+---+------+--------------+----------+
| id|income|spending_score|prediction|
+---+------+--------------+----------+
|  1| 20000|             3|         1|
|  2| 30000|             4|         1|
|  3| 40000|             2|         1|
|  4| 50000|             5|         0|
|  5| 60000|             7|         0|
|  6| 70000|             8|         0|
+---+------+--------------+----------+

Cluster Centers:
Cluster 0: [6.00000000e+04 6.66666667e+00]
Cluster 1: [3.e+04 3.e+00]
